In [ ]:
import torch
from torch import nn
from typing import Optional


1/torch.logspace(start=torch.log10(torch.tensor(0.1)), 
                 end=torch.log10(torch.tensor(20)), steps=10)

tensor([10.0000,  5.5505,  3.0808,  1.7100,  0.9491,  0.5268,  0.2924,  0.1623,
         0.0901,  0.0500])

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
sys.path.insert(0, str(project_root / "scripts"))

from torch.utils.data import DataLoader
from data import CanonicalParquetDataset
from model import FoundationNMRProcessor, H_AVAILABILITY_FIELDS
from scripts.model.utils.fourier_features import FourierFeatures

dataset_path = project_root / "datasets/cleaned/test_benchmark.parquet"
assert dataset_path.is_file(), f"Dataset non trovato: {dataset_path}"

dataset = CanonicalParquetDataset(dataset_path)
processor = FoundationNMRProcessor()

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=processor,
)

batch = next(iter(loader))

print("Record IDs:")
print(batch["record_ids"])

print("\nTensor shapes:")
for modality in ("h", "c"):
    for name, tensor in batch[modality].items():
        print(f"{modality}.{name:18s} {tuple(tensor.shape)}  {tensor.dtype}")

print("\nNumero di picchi reali per record:")
print("H:", batch["h"]["peak_mask"].sum(dim=1).tolist())
print("C:", batch["c"]["peak_mask"].sum(dim=1).tolist())

print("\nAvailability H:")
for channel, name in enumerate(H_AVAILABILITY_FIELDS):
    available = batch["h"]["availability"][:, :, channel].sum(dim=1)
    print(f"{name:18s}: {available.tolist()}")

# Ispeziona i valori reali del primo record.
h_mask = batch["h"]["peak_mask"][0]
c_mask = batch["c"]["peak_mask"][0]

print("\nPrimo record:", batch["record_ids"][0])
print("H shifts:", batch["h"]["shift"][0][h_mask].tolist())
print("C shifts:", batch["c"]["shift"][0][c_mask].tolist())
print("H integration:", batch["h"]["integration"][0][h_mask].tolist())
print("H multiplicity IDs:", batch["h"]["multiplicity"][0][h_mask].tolist())
print("H range half-span:", batch["h"]["range_half_span"][0][h_mask].tolist())

assert batch["h"]["shift"].shape == (8, 60)
assert batch["h"]["j_values"].shape == (8, 60, 6)
assert batch["h"]["availability"].shape == (8, 60, 4)
assert batch["c"]["shift"].shape == (8, 60)

print("\n✓ Primo batch reale processato correttamente")

Record IDs:
['nmrpeak-mst-nmr:test:10034', 'nmrpeak-mst-nmr:test:10037', 'nmrpeak-mst-nmr:test:10044', 'nmrpeak-mst-nmr:test:10051', 'nmrpeak-mst-nmr:test:1008', 'nmrpeak-mst-nmr:test:10085', 'nmrpeak-mst-nmr:test:1009', 'nmrpeak-mst-nmr:test:10112']

Tensor shapes:
h.shift              (8, 60)  torch.float32
h.integration        (8, 60)  torch.float32
h.multiplicity       (8, 60)  torch.int64
h.j_values           (8, 60, 6)  torch.float32
h.range_half_span    (8, 60)  torch.float32
h.peak_mask          (8, 60)  torch.bool
h.j_mask             (8, 60, 6)  torch.bool
h.availability       (8, 60, 4)  torch.bool
c.shift              (8, 60)  torch.float32
c.peak_mask          (8, 60)  torch.bool

Numero di picchi reali per record:
H: [7, 5, 12, 9, 13, 15, 9, 14]
C: [16, 10, 24, 27, 17, 18, 12, 22]

Availability H:
integration       : [7, 5, 12, 9, 13, 15, 9, 14]
multiplicity      : [7, 5, 12, 9, 13, 15, 9, 14]
j_values          : [7, 5, 12, 9, 13, 15, 9, 14]
range_half_span   : [7, 5, 12,

In [33]:
H_AVAILABILITY_FIELDS

('integration', 'multiplicity', 'j_values', 'range_half_span')

In [19]:
batch['h'].keys()

dict_keys(['shift', 'integration', 'multiplicity', 'j_values', 'range_half_span', 'peak_mask', 'j_mask', 'availability'])

In [20]:
batch['c'].keys()

dict_keys(['shift', 'peak_mask'])

In [30]:
_, h_peaks, c_peaks = batch.values()
h_shift, h_integration, h_multiplicity, h_j_values, h_range_half_span, h_peak_mask, h_j_mask, h_availability = h_peaks.values()

In [54]:
a_i, a_m, a_j, a_w = torch.unbind(h_availability, dim=-1)

In [40]:
embeddings = torch.ones((8, 60, 64))
a = h_availability[..., 0]
(a.unsqueeze(-1) * embeddings).shape

torch.Size([8, 60, 64])

In [57]:
embeddings_i = torch.ones((8, 60, 64))
embeddings_m = torch.ones((8, 60, 32))
embeddings_w = torch.ones((8, 60, 32))

torch.concat((embeddings_i, embeddings_m, embeddings_w, embeddings_w, a_i.unsqueeze(-1), a_m.unsqueeze(-1), a_w.unsqueeze(-1)), dim=-1).shape

torch.Size([8, 60, 163])

In [59]:
h_availability.shape

torch.Size([8, 60, 4])

In [62]:
a_rich = torch.argmax(h_availability, dim=-1)
a_rich

RuntimeError: "argmax_cpu" not implemented for 'Bool'

In [ ]:
class ShiftEmbedder(nn.Module):
    def __init__(self, x_min, x_max):

        self.fourier = FourierFeatures(
            strategy=self.fourier_strategy, 
            x_min=x_min, 
            x_max=x_max, 
            num_freqs=self.num_fourier_freqs)
        
        self.mlp = nn.Sequential(
                nn.Linear(self.num_fourier_freqs, 512),
                nn.GELU(),
                nn.Linear(512, self.d_model),
            )
    def forward(self):
        pass

class IntegrationEmbedder(nn.Module):
    def __init__(self):
        pass
    def forward(self):
        pass

class MultiplicityEmbedder(nn.Module):
    def __init__(self):
        pass
    def forward(self):
        pass

class JCouplingEmbedder(nn.Module):
    def __init__(self):
        pass
    def forward(self):
        pass

class WidthEmbedder(nn.Module):
    def __init__(self):
        pass
    def forward(self):
        pass

class HPeakEmbedder(nn.Module):
    def __init__(self):
        self.shift_embedder = ShiftEmbedder(*self.h_range)
        self.integration_embedder = IntegrationEmbedder()
        self.multiplicity_embedder = MultiplicityEmbedder()
        self.j_embedder = JCouplingEmbedder()
        self.width_embedder = WidthEmbedder()

        self.rich_mlp = nn.Sequential(
                nn.Linear(164, 256),
                nn.GELU(),
                nn.Linear(256, self.d_model),
            )

        self.layer_norm = nn.LayerNorm(self.d_model)

    def forward(
        self,
        shift: torch.Tensor,
        integration: Optional[torch.Tensor] = None,
        multiplicity: Optional[torch.Tensor] = None,
        j_values: Optional[torch.Tensor] = None,
        width: Optional[torch.Tensor] = None,
        j_mask: Optional[torch.Tensor] = None,
        availability: Optional[torch.Tensor] = None,
    ):
        a_i, a_m, a_j, a_w = torch.unbind(availability, dim=-1)

        shift_embedd = self.shift_embedder(shift)

        integration_embedd = a_i.unsqueeze(-1) * self.integration_embedder(integration)
        multiplicity_embedd = a_m.unsqueeze(-1) * self.multiplicity_embedder(multiplicity)
        j_embedd = a_j.unsqueeze(-1) * self.j_embedder(j_values, j_mask)
        width_embedd = a_w.unsqueeze(-1) * self.width_embedder(width)

        rich_input = torch.concat((
            integration_embedd, 
            multiplicity_embedd, 
            j_embedd, 
            width_embedd, 
            a_i.unsqueeze(-1), 
            a_m.unsqueeze(-1), 
            a_j.unsqueeze(-1), 
            a_w.unsqueeze(-1)))

        a_rich = h_availability.any(dim=-1)

        rich = self.rich_mlp(rich_input)
        rich = a_rich * rich

        peak_embedd = self.layer_norm(shift_embedd + rich)

        return peak_embedd

class CPeakEmbedder(nn.Module):
    def __init__(self):
        self.shift_embedder = ShiftEmbedder()
        self.layer_norm = nn.LayerNorm(512)

    def forward(     
        self,
        shift: torch.Tensor,
    ):

        shift_embedd = self.shift_embedder(shift)
        shift_embedd = self.layer_norm(shift_embedd)

        return shift_embedd


class NmrEmbedding(nn.Module):
    def __init__(
    self,
    d_model=512,
    dropout=0.1,
    fourier_strategy='log_spaced',     # Default strategy: lin_float_int
    h_x_min=0.01, h_x_max=20.0,
    c_x_min=0.1, c_x_max=300.0,
    num_fourier_freqs=256,      # If None, compute automatically
    ):
        self.h_range = (h_x_min, h_x_max)
        self.c_range = (c_x_min, c_x_max)
        self.num_fourier_freqs = num_fourier_freqs
        self.fourier_strategy = fourier_strategy
        self.d_model = d_model
        self.H_embedder = HPeakEmbedder()
        self.C_embedder = CPeakEmbedder()


    def forward(self, batch):
        _, h_peaks, c_peaks = batch.values()
        h_shift, h_integration, h_multiplicity, h_j_values, h_range_half_span, h_peak_mask, h_j_mask, h_availability = h_peaks.values()
        c_shift, c_peak_mask = c_peaks.values()
        h_tokens = self.H_embedder(
            shift=h_shift,
            integration=h_integration,
            multiplicity=h_multiplicity,
            j_values=h_j_values,
            width=h_range_half_span,
            j_mask=h_j_mask,
            availability=h_availability,
        )

        c_tokens = self.C_embedder(
            shift=c_shift,
        )

        h_tokens += h_type_embedding
        c_tokens += c_type_embedding

        tokens = concat_available(h_tokens, c_tokens)

        tokens = spectrum_transformer(
            tokens,
            padding_mask=peak_padding_mask,
        )

        spectrum_embedding = pool(tokens)
